In [44]:
import os
os.environ["SARVAM_API_KEY"] = "sk_5lfbky5q_bisrwBGPHDkHHb5VHSXlznDV"

## Install libraries

In [45]:
!pip install -q youtube-transcript-api langchain-community langchain-openai \
               faiss-cpu tiktoken python-dotenv sentence-transformers

In [46]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

## Step 1a - Indexing (Document Ingestion)

In [47]:
video_id = "OcISVEh1jyw" # only the ID, not full URL
try:
    # If you don’t care which language, this returns the “best” one
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=["en"])

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

when I was younger um it was very romanticized that you're going for it and just keep going and you know it's like you have to work 24 hours do four movies in one time you know five shifts don't sleep don't eat and the more a person does that the more successful you will be but as I have seen and lived more life I've realized having a work life balance is really important so when you talk about a beach I was in Turks and Kos just recently my phones were off me and my husband just by ourselves we spent time we walk walked on the beach we collected shells um and then I came back feeling rejuvenated feeling excited and really motivated to take my career forward I mean this generation of girls is fearless and there's so many girls that take charge of their own lives and they say you know I'm not going to fit into the cookie cutter mold that probably my mother did or my grandmother had to and it's so like I have Goosebumps right now just thinking about it I meet so many young women who have

In [48]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='when I was younger um it was very', start=0.08, duration=4.16), FetchedTranscriptSnippet(text="romanticized that you're going for it", start=2.48, duration=3.72), FetchedTranscriptSnippet(text="and just keep going and you know it's", start=4.24, duration=4.0), FetchedTranscriptSnippet(text='like you have to work 24 hours do four', start=6.2, duration=4.96), FetchedTranscriptSnippet(text='movies in one time you know five shifts', start=8.24, duration=5.559), FetchedTranscriptSnippet(text="don't sleep don't eat and the more a", start=11.16, duration=4.639), FetchedTranscriptSnippet(text='person does that the more successful you', start=13.799, duration=4.56), FetchedTranscriptSnippet(text='will be but as I have seen and lived', start=15.799, duration=4.681), FetchedTranscriptSnippet(text="more life I've realized having a work", start=18.359, duration=4.281), FetchedTranscriptSnippet(text='life balance is really important so when'

## Step 1b - Indexing (Text Splitting)

In [49]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [50]:
len(chunks)

60

In [51]:
chunks[30]

Document(metadata={}, page_content="people don't think about how much hard work how much consistent knocking on doors how much humility it takes to go to another country and start from scratch I had built an incredible career here an amazing credibility here and um I got an opportunity to do music and people were curious about the fact that cuz I could sing what that could be and I'm a big fan of the music industry anyway we um in IND monopolized by Bollywood when it comes to music so much for now I hope so I hope so that we've not really been able to bate into pop you know pop music or pop culture when it comes to music it's still very Niche so I was really a big fan of like music and musicians and um very excited about that opportunity went there worked with the most incredible musicians and quickly realized that I'm not as good as them you know um and I should go back to my day job I don't like I'm very um astute when it comes to I don't lie to myself I'm very honest to myself so I 

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [52]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

C:\Users\Sadguru\AppData\Local\Temp\ipykernel_12380\4220134732.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
C:\Users\Sadguru\AppData\Roaming\Python\Python311\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [55]:
vector_store.index_to_docstore_id

{0: 'aa0f5ba2-29ee-4054-8949-42aa740a4ba6',
 1: '1fcedca3-496c-43da-8b71-3d5d1746f022',
 2: 'f052c65d-efe2-40a3-af50-6686aa1311fc',
 3: '766fbdf8-29d2-4e66-9c18-b71d98fed449',
 4: '7e4ca1ac-26b7-4dac-9cb4-9636cd169b63',
 5: '1acc94e6-f443-4ebe-9081-a6c94d2a2684',
 6: 'a2db590f-4e75-44e7-aef4-581d998c1c16',
 7: 'f0f5d65b-388d-46a9-a3c9-f7fd05099be8',
 8: 'ec5cf95b-ece2-4207-bec7-78b67cc13ff4',
 9: 'a9ff340a-3a1e-4713-9641-31276b8839a5',
 10: '37ae50de-2e49-4efd-8c9d-57f233898443',
 11: 'ba1b7900-cdcc-44cf-a9eb-8a6b3a7ba59b',
 12: '07369b60-a5b7-4c3f-b643-c9bf0eb5bb66',
 13: '08d3ab4d-8566-4061-836d-33ed7e8ef3a6',
 14: '16010e07-6461-4998-800d-aafaaada2602',
 15: '3af85cb0-a917-498a-b74d-886f156f0544',
 16: '1d02af49-bbd6-43c8-a406-d8ff6c087d99',
 17: '49434b9a-860e-4e08-95c2-7a6e52836b44',
 18: '4fa290a1-bcc0-4c85-9960-1886a0a772eb',
 19: 'c787de8b-2d6d-42e9-8ab2-11cee2edb55f',
 20: 'dcc3fd4e-a1a5-437e-b74f-0ef9943fbd2a',
 21: '52c41004-1b84-4cfa-8da2-af8dde8d2edb',
 22: '43088584-9990-

In [58]:
# FAISS does not support get_by_ids directly.
# Instead, we can get an ID from the mapping and search the docstore:
sample_id = list(vector_store.index_to_docstore_id.values())[0]
vector_store.docstore.search(sample_id)


NotImplementedError: FAISS does not yet support get_by_ids.

## Step 2 - Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7fdba029d2d0>, search_kwargs={'k': 4})

In [ ]:
retriever.invoke("Priyanka's Entrepreneurial Mindset")

[Document(id='7c0f7504-08fa-4f33-8708-d29bfc601f84', metadata={}, page_content="the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal qu

## Step 3 - Augmentation

In [ ]:
llm = ChatOpenAI(
    model="sarvam-105b", 
    temperature=0.2, 
    api_key=os.environ.get("SARVAM_API_KEY"), 
    base_url="https://api.sarvam.ai/v1"
)


In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question          = "is the topic of dark effects of being famous  discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [ ]:
retrieved_docs

[Document(id='3c60d0d6-5d01-4dfc-99fc-5c4bb4422cb0', metadata={}, page_content="so we with this problem and we published it in a nature paper last year uh we held the fusion that we held the plasma in specific shapes so actually it's almost like carving the plasma into different shapes and control and hold it there for the record amount of time so um so that's one of the problems of of fusion sort of um solved so i have a controller that's able to no matter the shape uh contain it continue yeah contain it and hold it in structure and there's different shapes that are better for for the energy productions called droplets and and and so on so um so that was huge and now we're looking we're talking to lots of fusion startups to see what's the next problem we can tackle uh in the fusion area so another fascinating place in a paper title pushing the frontiers of density functionals by solving the fractional electron problem so you're taking on modeling and simulating the quantum mechanical 

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"so we with this problem and we published it in a nature paper last year uh we held the fusion that we held the plasma in specific shapes so actually it's almost like carving the plasma into different shapes and control and hold it there for the record amount of time so um so that's one of the problems of of fusion sort of um solved so i have a controller that's able to no matter the shape uh contain it continue yeah contain it and hold it in structure and there's different shapes that are better for for the energy productions called droplets and and and so on so um so that was huge and now we're looking we're talking to lots of fusion startups to see what's the next problem we can tackle uh in the fusion area so another fascinating place in a paper title pushing the frontiers of density functionals by solving the fractional electron problem so you're taking on modeling and simulating the quantum mechanical behavior of electrons yes um can you explain this work and can ai model and\n\n

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      so we with this problem and we published it in a nature paper last year uh we held the fusion that we held the plasma in specific shapes so actually it's almost like carving the plasma into different shapes and control and hold it there for the record amount of time so um so that's one of the problems of of fusion sort of um solved so i have a controller that's able to no matter the shape uh contain it continue yeah contain it and hold it in structure and there's different shapes that are better for for the energy productions called droplets and and and so on so um so that was huge and now we're looking we're talking to lots of fusion startups to see what's the next problem we can tackle uh in the fusion area so another fascinating place in a paper title pushing the frontiers of density functionals

## Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of nuclear fusion is discussed in the video. The discussion includes the following points:

1. The speaker mentions a problem in fusion that was published in a Nature paper, where they developed a controller that can hold plasma in specific shapes for a record amount of time, which is crucial for energy production.

2. They talk about collaborating with EPFL in Switzerland, which has a test reactor that they used for their experiments. The focus is on identifying bottleneck problems in fusion and applying AI methods to address those challenges.

3. The speaker emphasizes the potential of AI to help accelerate solutions in energy and climate, specifically mentioning fusion as an area where AI can contribute.

4. They also mention their work on magnetic control of tokamak plasmas using deep reinforcement learning, indicating that they are exploring how AI can assist in controlling high-temperature plasmas for nuclear fusion.


## Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('who is Priyanka Chopra')

{'context': "the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')

'The video features a conversation with Demas, who discusses the need for deeper and simpler explanations in physics, particularly in relation to consciousness, life, and gravity. He emphasizes the limitations of the current standard model of physics and the importance of exploring more fundamental explanations. Additionally, he talks about advancements in fusion research, specifically how they have managed to hold plasma in specific shapes for extended periods, which is a significant step in fusion energy production. The conversation touches on the potential of AI in modeling quantum mechanical behavior, particularly regarding electrons. The discussion concludes with a quote about the nature of computer science.'